In [1]:
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 77.3 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


In [2]:
import pdfplumber
import pandas as pd
import numpy as np
import re
import joblib

from sklearn.metrics.pairwise import cosine_similarity

In [3]:
def extract_pdf_text(pdf_path):

    text = ""

    with pdfplumber.open(pdf_path) as pdf:

        for page in pdf.pages:

            page_text = page.extract_text()

            if page_text:
                text += page_text

    return text

In [4]:
from google.colab import files

uploaded = files.upload()

Saving Suvalakshmi_Resume.pdf to Suvalakshmi_Resume.pdf


In [5]:
pdf_file = list(uploaded.keys())[0]

pdf_file

'Suvalakshmi_Resume.pdf'

In [6]:
resume_text = extract_pdf_text(pdf_file)

print(resume_text[:500])

SUVALAKSHMI S
suvalakshmis01@gmail.com | 9360518741 | Kovilpatti, Tuticorin, TN | LinkedIn Profile
PROFESSIONAL SUMMARY
Dynamic and motivated M.E. graduate in Biometrics and Cyber Security with strong expertise in deep learning, image processing,
and blockchain-based security systems. Proficient in Python and C, with practical knowledge of malware analysis, network
security, and penetration testing tools. Demonstrated ability to design and implement innovative, research-driven solutions for real


In [7]:
def clean_text(text):

    text = text.lower()

    text = re.sub(
        r'http\S+|www\S+',
        '',
        text
    )

    text = re.sub(
        r'\S+@\S+',
        '',
        text
    )

    text = re.sub(
        r'[^a-z\s]',
        ' ',
        text
    )

    text = re.sub(
        r'\s+',
        ' ',
        text
    )

    return text.strip()

In [8]:
clean_resume = clean_text(resume_text)

clean_resume[:300]

'suvalakshmi s kovilpatti tuticorin tn linkedin profile professional summary dynamic and motivated m e graduate in biometrics and cyber security with strong expertise in deep learning image processing and blockchain based security systems proficient in python and c with practical knowledge of malware'

In [13]:
classifier = joblib.load(
    "/content/drive/MyDrive/AI-Career-Assistant/data/raw/models/career_classifier.pkl"
)

In [14]:
tfidf = joblib.load(
    "/content/drive/MyDrive/AI-Career-Assistant/data/raw/models/tfidf_vectorizer.pkl"
)

In [15]:
resume_vector = tfidf.transform(
    [clean_resume]
)

In [16]:
prediction = classifier.predict(
    resume_vector
)

prediction[0]

'Data Science'

In [18]:
jobs = pd.read_csv(
    "/content/drive/MyDrive/AI-Career-Assistant/data/processed/clean_jobs.csv"
)

In [19]:
job_vectorizer = joblib.load(
    "/content/drive/MyDrive/AI-Career-Assistant/data/raw/models/job_vectorizer.pkl"
)

In [20]:
job_vectors = job_vectorizer.transform(
    jobs["Clean_Skills"]
)

In [21]:
def recommend_jobs(resume_text, top_n=5):

    resume_vector = job_vectorizer.transform(
        [resume_text]
    )

    scores = cosine_similarity(
        resume_vector,
        job_vectors
    )[0]


    indexes = scores.argsort()[::-1][:top_n]


    result = jobs.iloc[indexes][
        ["Job_Title"]
    ].copy()


    result["Match"] = (
        scores[indexes] * 100
    ).round(2)


    return result

In [22]:
recommend_jobs(clean_resume)

,Job_Title,Match
2,AI Engineer,41.84
1,Machine Learning Engineer,40.91
5,NLP Engineer,37.07
3,Data Analyst,29.87
0,Data Scientist,14.03


In [23]:
def extract_skills(text):

    skills = [
        "python",
        "sql",
        "machine learning",
        "tensorflow",
        "aws",
        "docker",
        "kubernetes",
        "nlp",
        "pandas",
        "numpy"
    ]

    found=[]

    for skill in skills:

        if skill in text:
            found.append(skill)

    return found

In [24]:
def analyze_resume(pdf_path):

    text = extract_pdf_text(
        pdf_path
    )

    clean = clean_text(
        text
    )


    role = classifier.predict(
        tfidf.transform([clean])
    )[0]


    jobs_result = recommend_jobs(
        clean
    )


    skills = extract_skills(
        clean
    )


    return {
        "Predicted Role": role,
        "Skills": skills,
        "Recommended Jobs": jobs_result
    }

In [25]:
result = analyze_resume(
    pdf_file
)

result

{'Predicted Role': 'Data Science',
 'Skills': ['python'],
 'Recommended Jobs':                    Job_Title  Match
 2                AI Engineer  41.84
 1  Machine Learning Engineer  40.91
 5               NLP Engineer  37.07
 3               Data Analyst  29.87
 0             Data Scientist  14.03}

In [27]:
report = pd.DataFrame(
    {
        "Predicted Role":
        [result["Predicted Role"]],

        "Skills":
        [", ".join(result["Skills"])]
    }
)


report.to_csv(
    "/content/drive/MyDrive/AI-Career-Assistant/reports/pdf_resume_results.csv",
    index=False
)

print("PDF analysis report saved")

PDF analysis report saved


In [28]:
!find reports -type f

find: ‘reports’: No such file or directory


In [29]:
!git status

fatal: not a git repository (or any of the parent directories): .git


In [30]:
%cd /content/drive/MyDrive/AI-Career-Assistant

/content/drive/MyDrive/AI-Career-Assistant


In [31]:
!git status

Refresh index: 100% (38/38), done.
On branch main
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/09_Skill_Gap_Analysis.ipynb
	modified:   notebooks/11_PDF_Resume_Upload_System.ipynb

no changes added to commit (use "git add" and/or "git commit -a")


In [32]:
import json

path = "notebooks/11_PDF_Resume_Upload_System.ipynb"

try:
    with open(path, "r", encoding="utf-8") as f:
        json.load(f)
    print("Notebook is valid")
except Exception as e:
    print(e)

Notebook is valid
